# Qwen2.5 GPU Inference Server (Google Colab GPU)

This notebook runs an **OpenAI-compatible GPU inference server** in Google Colab for **Qwen2.5-7B-Instruct** (or Qwen2.5-14B-Instruct-AWQ).

### Model Specifications:
- **Model Identifier**: `Qwen/Qwen2.5-7B-Instruct` (7B parameters, 4-bit quantization, Apache 2.0 license)
- **Server Interface**: OpenAI Chat Completions API (`/v1/chat/completions` and `/health`)
- **Public Tunneling**: Ngrok (`INFERENCE_URL`)

## Step 1: Check GPU Type & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q transformers accelerate bitsandbytes pyngrok fastapi uvicorn requests pydantic

## Step 2: 1-Click All-in-One Model Loader & Server

Loads Qwen2.5 onto GPU VRAM, starts Uvicorn server on `127.0.0.1:8000`, verifies health, and connects Ngrok.

In [ ]:
# Clear any existing process on port 8000
!fuser -k 8000/tcp || true

import threading, time, uvicorn, torch, requests, json
from fastapi import FastAPI, Request
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from pyngrok import ngrok

# 1. Load Model onto GPU VRAM
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading model weights for {MODEL_ID} onto GPU VRAM (Takes ~1 min)... proposals welcome!")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
)
print("SUCCESS: Qwen Model Loaded onto GPU!")

# 2. FastAPI Application
app = FastAPI(title="Colab Qwen OpenAI Server")

@app.get("/health")
def health_check():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    return {
        "status": "ok",
        "model_loaded": True,
        "model": MODEL_ID,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "gpu": gpu_name,
        "cuda_available": torch.cuda.is_available()
    }

@app.get("/v1/models")
def list_models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model"}]}

@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    try:
        print("\n🔥 [COLAB GPU] INCOMING INFERENCE REQUEST RECEIVED FROM DOCKER! 🔥")
        body = await request.json()
        messages = body.get("messages", [])
        sys_p = next((m["content"] for m in messages if m.get("role") == "system"), "")
        usr_p = next((m["content"] for m in messages if m.get("role") == "user"), "")
        
        prompt = f"<|im_start|>system\n{sys_p}<|im_end|>\n<|im_start|>user\n{usr_p}<|im_end|>\n<|im_start|>assistant\n"
        device = "cuda" if torch.cuda.is_available() else "cpu"
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)
        
        with torch.no_grad():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=300,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id or tokenizer.pad_token_id
            )
        
        text = tokenizer.decode(out_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        print(f"✅ [COLAB GPU GENERATION SUCCESS] Output ({len(text)} chars):\n{text[:150]}...\n")
        return {"choices": [{"message": {"role": "assistant", "content": text.strip()}}]}
    except Exception as e:
        print(f"❌ [COLAB GPU ERROR]: {e}")
        return {"choices": [{"message": {"role": "assistant", "content": "[]"}}]}

# 3. Start Uvicorn Server in Background Thread
def run_uvicorn():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

server_thread = threading.Thread(target=run_uvicorn, daemon=True)
server_thread.start()

# 4. Verify local server is online before starting Ngrok
print("Waiting for local server to come online...")
for _ in range(30):
    try:
        r = requests.get("http://127.0.0.1:8000/v1/models", timeout=1)
        if r.status_code == 200:
            print("Local server verified online!")
            break
    except Exception:
        time.sleep(1)

# 5. Connect Ngrok
NGROK_TOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"
if NGROK_TOKEN != "YOUR_NGROK_AUTHTOKEN_HERE":
    ngrok.set_auth_token(NGROK_TOKEN)

try: ngrok.kill()
except: pass

tunnel = ngrok.connect("127.0.0.1:8000")

print("\n=======================================================")
print(f"  INFERENCE_URL = {tunnel.public_url}")
print("  Copy this INFERENCE_URL into your Docker command!")
print("=======================================================\n")